In [1]:
import scanpy as sc
import scarches as sca
# import matplotlib.pyplot as plt
import numpy as np
import gdown
import pandas as pd
import pandas as pd
import scipy.sparse as sp

 captum (see https://github.com/pytorch/captum).


In [2]:
# Load the AnnData object and SCANVI model (use exact paths from epithelial_extended_atlas_integration.py)
adata_hvg_epi = sc.read_h5ad('/lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/healthy_exocrine_integration_extended_atlas_500.h5ad')
scanvae = sca.models.SCANVI.load('/lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_extended_atlas_500', adata=adata_hvg_epi)

INFO     File                                                                                                      
         /lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_exten
         ded_atlas_500/model.pt already downloaded                                                                 


/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/aih/shrey.parikh/miniconda3/envs/scarches/lib/ ...


In [3]:
print("Mapping PDAC to Extended Atlas")
pdac = sc.read_h5ad('/lustre/groups/ml01/workspace/shrey.parikh/PDAC_Work_Dir/PDAC_Final/Human_Atlas_Harmonised_genes_filtered.h5ad')
epi_pdac = pdac[pdac.obs.Level_4.str.contains('Acinar|Ductal|Malignant', na=False)]
epi_pdac_hvg = epi_pdac[:, epi_pdac.var_names.isin(adata_hvg_epi.var_names)]
del pdac
import gc
gc.collect()

Mapping PDAC to Extended Atlas


124

In [5]:
import anndata as ad

In [6]:

missing = [g for g in adata_hvg_epi.var_names if g not in epi_pdac_hvg.var_names]
if missing:
    print(f"Missing genes in epi_pdac_hvg: {missing}")
    X_missing = sp.csr_matrix((epi_pdac_hvg.n_obs, len(missing)))
    X_new = sp.hstack([epi_pdac_hvg.X, X_missing], format="csr")
    var_missing = pd.DataFrame(index=missing, columns=epi_pdac_hvg.var.columns)
    var_new = pd.concat([epi_pdac_hvg.var.copy(), var_missing], axis=0)
    epi_pdac_hvg = ad.AnnData(
        X=X_new,
        obs=epi_pdac_hvg.obs.copy(),
        var=var_new
    )

Missing genes in epi_pdac_hvg: ['LINC02789']


In [7]:

epi_pdac_hvg = epi_pdac_hvg[:, adata_hvg_epi.var_names].copy()
epi_pdac_hvg.layers['counts'] = epi_pdac_hvg.X.copy()
epi_pdac_hvg.obs['batch_covar_split'] = epi_pdac_hvg.obs['Dataset_ID'].astype(str).values
epi_pdac_hvg.obs.rename(columns={'Level_4': 'Level_4_PDAC'}, inplace=True)

In [12]:
!nvidia-smi

No devices were found


In [9]:

model = sca.models.SCANVI.load_query_data(epi_pdac_hvg, '/lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_extended_atlas_500', 
                                          freeze_dropout = True)
model._unlabeled_indices = np.arange(epi_pdac_hvg.n_obs)
model._labeled_indices = []
print("Labelled Indices: ", len(model._labeled_indices))
print("Unlabelled Indices: ", len(model._unlabeled_indices))
model.train(max_epochs=40, plan_kwargs=dict(weight_decay=0.0), check_val_every_n_epoch=10)

INFO     File                                                                                                      
         /lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_exten
         ded_atlas_500/model.pt already downloaded                                                                 


/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/aih/shrey.parikh/miniconda3/envs/scarches/lib/ ...
/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 149 in adata.obs['_scvi_batch'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(


Labelled Indices:  0
Unlabelled Indices:  481924
INFO     Training for 40 epochs.                                                                                   


/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/aih/shrey.parikh/miniconda3/envs/scarches/lib/ ...
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python

Training:   0%|          | 0/40 [00:00<?, ?it/s]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


In [ ]:
epi_pdac.obs['predictions'] = model.predict()
adata_pdac = adata_epi.concatenate(epi_pdac, batch_key="condition",batch_categories=["Healthy", "PDAC"])
adata_pdac.obs["Level_4_predictions"] = adata_pdac.obs["Level_4"].astype(object)
pdac_mask = adata_pdac.obs["condition"] == "PDAC"
adata_pdac.obs.loc[pdac_mask, "Level_4_predictions"] = adata_pdac.obs.loc[pdac_mask, "predictions"]
full_latent = model.get_latent_representation(adata=adata_pdac)
adata_pdac.obsm['scanvi_extended_pdac'] = full_latent.copy()
sc.pp.neighbors(adata_pdac, use_rep='scanvi_extended_pdac', n_neighbors=50,  metric='cosine')
sc.tl.umap(adata_pdac, min_dist=0.5)
for col in adata_pdac.var.columns:
    if adata_pdac.var[col].dtype == "object":
        adata_pdac.var[col] = adata_pdac.var[col].astype(str)

In [16]:
adata_hvg_epi

AnnData object with n_obs × n_vars = 538355 × 892
    obs: 'barcode', 'Dataset', 'ID', 'sample_id', 'donor_id', 'alignment_software', 'assay_ontology_term_id', 'institute', 'library_id', 'library_preparation_batch', 'library_sequencing_run', 'Manual_Annotation', 'cell_enrichment', 'condition', 'celltype_all', 'manner_of_death', 'reference_genome', 'sample_collection_method', 'sample_source', 'sampled_site_condition', 'samples', 'sex', 'sex--cell_ontology_term_id', 'suspension_type', 'technology', 'tissue_type', 'assay', 'tissue', 'organism', 'disease', 'gene_annotation_version', 'batch_covar_split', 'batch_covar_split_donor_id', 'batch_covariate', 'info_batch_covariate_split', 'doublet_score', 'predicted_doublet', 'n_genes', 'total_counts', 'total_counts_mito', 'total_counts_ribo', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'log1p_total_counts_mito', 'log1p_total_counts_ribo', 'n_genes_by_counts', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_50

In [13]:
epi_pdac_hvg

AnnData object with n_obs × n_vars = 481924 × 892
    obs: 'Sample_ID', 'Condition', 'Treatment', 'TreatmentType', 'TreatmentStatus', 'Tissue', 'Sex', 'Dataset', 'Technology', 'Level_1', 'Level_2', 'Level_3', 'Level_4_PDAC', 'Age', 'Diabetes', 'Is_Core', 'EMT category', 'Dataset_ID', 'Cluster_Names', 'mal_vs_healthy', 'neuronal_markers', 'mesenchymal_ecm_markers', 'er_stress_markers', 'oncogenic_markers', 'batch_covar_split', '_scvi_batch', 'Level_4', '_scvi_labels'
    var: 'n_cells', 'ensembl_id', 'start', 'end', 'chromosome', 'gene_name_adata_sc', 'highly_variable_adata_sc', 'means_adata_sc', 'dispersions_adata_sc', 'dispersions_norm_adata_sc', 'highly_variable_nbatches_adata_sc', 'highly_variable_intersection_adata_sc', 'n_cells_by_counts_adata_sc', 'mean_counts_adata_sc', 'log1p_mean_counts_adata_sc', 'pct_dropout_by_counts_adata_sc', 'total_counts_adata_sc', 'log1p_total_counts_adata_sc', 'mito_adata_sc', 'n_cells_by_counts_adata_sn', 'mean_counts_adata_sn', 'log1p_mean_counts_ad

In [15]:
model = sca.models.SCANVI.load(dir_path="/lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_extended_atlas_500", adata=adata_hvg_epi)
model = sca.models.SCANVI.load_query_data(epi_pdac_hvg, '/lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_extended_atlas_500', 
                                          freeze_dropout = True)
model._unlabeled_indices = np.arange(epi_pdac_hvg.n_obs)
model._labeled_indices = []
print("Labelled Indices: ", len(model._labeled_indices))
print("Unlabelled Indices: ", len(model._unlabeled_indices))
model.train(max_epochs=1, plan_kwargs=dict(weight_decay=0.0), check_val_every_n_epoch=10)
adata_concat = adata_hvg_epi.concatenate(epi_pdac_hvg, batch_key="condition",batch_categories=["Healthy", "PDAC"])
adata_emb = ad.AnnData(X=model.get_latent_representation(adata_concat))
adata_emb.obs = adata_concat.obs.copy()

INFO     File                                                                                                      
         /lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_exten
         ded_atlas_500/model.pt already downloaded                                                                 


/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/aih/shrey.parikh/miniconda3/envs/scarches/lib/ ...


INFO     File                                                                                                      
         /lustre/groups/ml01/workspace/hpca/hpca_downstream/Epithelial_PDAC_Extension/scanvi_model_epithelial_exten
         ded_atlas_500/model.pt already downloaded                                                                 


/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/aih/shrey.parikh/miniconda3/envs/scarches/lib/ ...
/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 149 in adata.obs['_scvi_batch'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(


Labelled Indices:  0
Unlabelled Indices:  481924
INFO     Training for 1 epochs.                                                                                    


/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/aih/shrey.parikh/miniconda3/envs/scarches/lib/ ...
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python

Training:   0%|          | 0/1 [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.
/tmp/ipykernel_3434653/3009444548.py:9: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_concat = adata_hvg_epi.concatenate(epi_pdac_hvg, batch_key="condition",batch_categories=["Healthy", "PDAC"])


INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/home/aih/shrey.parikh/miniconda3/envs/scarches/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 149 in adata.obs['_scvi_batch'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
